In [42]:
import numpy as np
import sympy as sym
from sympy import pprint


In [12]:
#Rough
def pp(a, b = 'Full', tolerance = 1e-12):
    import sympy as sym
    from sympy import pprint



    if b == 'Full':
        if type(a) == np.ndarray:
            c = sym.Matrix(a)
        else:
            print(a)
    else:
        c = np.round(a.real, b) + 1j*np.round(a.imag, b)
        c = np.where(np.abs(c.imag) < tolerance, c.real, c)
        c = sym.Matrix(c)
    pprint(c)



def pp(M, rounding=3, imag_tolerance = 1e-12):
    if type(M) is not np.ndarray:
        print(M)
    else:
        M = M.real.round(rounding) + 1j*M.imag.round(rounding)
        if imag_tolerance is not None:
            M = np.where(np.abs(M.imag) < imag_tolerance, M.real, M)
        print(np.round(M, rounding))


In [32]:
class QuantumSystem:
    def __init__(self, q, alpha = 'auto'):
        self.q = q      # q is the number of main qubits, there are extra q ancilla qubits . Total 2*q qubits
        self.N = 2**q
        self.alpha = alpha

        self.psi0_a = self.superposition_state()
        self.zero_b = self.superposition_state(coeffs = [1] + [0]*(self.N-1))
        self.psi_kron_0 = np.kron(self.psi0_a, self.zero_b)
        
        self.I_a = np.eye(self.N)
        self.I_b = np.eye(self.N)
        self.Proj_ab = np.kron(self.I_a, self.zero_b @ self.zero_b.conj().T)    # projection operator - Identity on a, projection on |0> on b
        
        self.Phase = np.diag(self.phase_operator_phases(alpha))                      # phase operator from the phases, "U_a" in notes
        self.Swap = self.U_ab()                                                 # swap operator, "U_ab" in notes
        
    def matrix_basis(self):
        q = self.q
        N = 2**q
        basis = []
        for i in range(N):
            state_i = np.zeros(N)
            state_i[i] = 1
            basis.append(state_i)
        return np.array(basis)

    def superposition_state(self, coeffs = 'maximal_except_0'):        # careful, dont include the zeros
        q = self.q
        N = self.N
        if coeffs == 'maximal_except_0':
            coeffs = np.ones(N)
            coeffs[0] = 0

        basis = self.matrix_basis()

        c = np.array(coeffs).reshape(1, N)
        c = c/np.sqrt(np.sum(np.abs(c)**2))  # normalizing 

        if basis.shape[1] != c.shape[1]:
            raise ValueError('Error: Shapes of coefficients and basis do not match')
        if len(coeffs) != N:            # checking length
            raise ValueError('Error: The number of coefficients is not 2^q')

        sup_state = np.sum(c * basis, axis = 1, keepdims = True)
        if (np.linalg.norm(sup_state)-1) > 1e-10:
            raise ValueError('Error: The resultant state is not normalized')
        return sup_state
    
    
    #defining the "partial swap" matrix for q qubits
    def U_ab(self):    
        q, N = self.q, self.N
        def kron_delta(a, b):
            return 1 if a == b else 0
        def U_mn(m,n):
            i, j = divmod(m, N)
            k, l = divmod(n, N)

            if k == 0 and l == 0:
                return (kron_delta(i, 0) * kron_delta(j, 0))
            elif k == 0 and l != 0:
                return (1/np.sqrt(2))*(kron_delta(i, 0) * kron_delta(j, l) - kron_delta(i, l) * kron_delta(j, 0))
            elif k != 0 and l == 0:
                return (1/np.sqrt(2))*(kron_delta(i, k) * kron_delta(j, 0) + kron_delta(i, 0) * kron_delta(j, k))
            else:
                return (kron_delta(i, k) * kron_delta(j, l))

        U = np.zeros((N**2, N**2))
        for m in range(N**2):       # create matrix using for loops - numpy gives errors
            for n in range(N**2):
                U[m, n] = U_mn(m, n)
        return U


    def phase_operator_phases(self, alpha = 'auto'):       # phases for the phase operator
        q, N = self.q, self.N
        # alpha is the coefficient in the Hamiltonian
        if q%2 == 0:            # creating alpha automatically, see notebook
            alpha = 4*np.pi/q**2
        else:
            alpha = 4*np.pi/(q**2-1)
        if alpha != 'auto':
            alpha = alpha           

        def qubit_sequence(a,q):
            b = np.binary_repr(a, width = q)       # string
            c = np.array([int(i) for i in b])   # array
            return c

        def edges_count(seq):   #seq is a binary sequence array
            p_0 = np.count_nonzero(seq == 0)
            p_1 = np.count_nonzero(seq == 1)
            return int(p_0*p_1)

        phases = []
        for i in range(N):
            cross_edges_i = edges_count(qubit_sequence(i, q)) 
            phase_i = np.exp(-1j*alpha*cross_edges_i)
            phases.append(phase_i)
        
        return np.array(phases)
        



In [46]:
Q2 = QuantumSystem(2)

pp(Q2.Phase,1)


# np.eye(4)


⎡1.0   0     0     0 ⎤
⎢                    ⎥
⎢ 0   -1.0   0     0 ⎥
⎢                    ⎥
⎢ 0    0    -1.0   0 ⎥
⎢                    ⎥
⎣ 0    0     0    1.0⎦


In [25]:
import numpy as np

# Given adjacency matrix
graph = np.array([
    [0, 1, 1, 0],  # A
    [1, 0, 0, 1],  # B
    [1, 0, 0, 1],  # C
    [0, 1, 1, 0]   # D
])

# Use np.where to get indices where the value is 1
indices = np.where(graph < 2)

# Stack the row and column indices to create a list of tuples
indices_of_ones = np.stack(indices, axis=-1)

# print(indices_of_ones)

indices_of_ones.shape


ValueError: operands could not be broadcast together with shapes (16,2) (4,4) 

In [15]:
print(indices_of_ones)

[[0 0]
 [0 1]
 [0 2]
 [0 3]
 [1 0]
 [1 1]
 [1 2]
 [1 3]
 [2 0]
 [2 1]
 [2 2]
 [2 3]
 [3 0]
 [3 1]
 [3 2]
 [3 3]]


In [26]:
per = np.array([1,0,1,0,1,1])
# create arrays having indices of 1s and 0s
ones =  np.array(np.where(per == 1))
zeros = np.array(np.where(per == 0))
ones.shape
# array having cartesian product of 1s and 0s elements
# cartesian_product = np.array(np.meshgrid(ones, zeros))
# cartesian_product

(1, 4)

In [33]:
arr = np.array([[0, 2, 3], [4, 5, 6], [7, 8, 9]])

a = arr.copy()



In [38]:
# a[a < 5] += 10
b = np.where(a > -10)

b

(array([0, 0, 0, 1, 1, 1, 2, 2, 2]), array([0, 1, 2, 0, 1, 2, 0, 1, 2]))

In [3]:
import numpy as np
qubit_string = '101'
qubit_perm = np.array([int(i) for i in qubit_string])

qubit_perm[qubit_perm == 0] = -1

col_indices, row_indices = np.meshgrid(qubit_perm, qubit_perm)  # eg. for permutation 101, col_indices will have columns 1,0 and 1; row_indices will have rows 1,0 and 1. Multiplying these will give only the cross edges to be 0, remaining will be 1. 

cross_edges_raw = col_indices * row_indices
cross_edges_ut_raw = np.triu(cross_edges_raw, k = 1)        # upper triangular part of the matrix to avoid double counting
cross_edges_ut_raw[cross_edges_ut_raw == 1] = 0             # inverting, to change cross edges from 0 to -1 (and latter to 1)
cross_edges = (-1)*cross_edges_ut_raw                       # changing cross edges from -1 to 1

cross_edges_count = np.sum(cross_edges)

cross_edges ,cross_edges_count



(array([[0, 1, 0],
        [0, 0, 1],
        [0, 0, 0]]),
 2)

In [6]:
import numpy as np

def adjacency_to_edge_array(adj_matrix):
    """
    Converts an adjacency matrix to a 2D NumPy array of edges.
    """
    origin, destination = np.nonzero(adj_matrix)
    edge_array = np.vstack((origin, destination)).T
    
    return edge_array           # number of rows are twice the number of edges coz of repetition



adj_sq = np.array([[0, 1, 1, 0], [1, 0, 0, 1], [1, 0, 0, 1], [0, 1, 1, 0]])



array([[0, 1],
       [0, 2],
       [1, 0],
       [1, 3],
       [2, 0],
       [2, 3],
       [3, 1],
       [3, 2]])